# データ読込

In [1]:
# ローカルユーティリティ: 外部ディレクトリ(utils, logic)への依存を排除
import pandas as pd
import sqlite3
from datetime import datetime, date, timedelta
from typing import Union, List
import jpholiday

# 日本の祝日取得（jpholiday が無ければ空集合）
def get_japanese_holidays(
    start: Union[str, date], end: Union[str, date], as_str: bool = True
) -> Union[List[str], List[date]]:
    """
    指定した期間の日本の祝日を取得する関数。

    Args:
        start (str or date): 開始日（"YYYY-MM-DD" または date型）
        end (str or date): 終了日（"YYYY-MM-DD" または date型）
        as_str (bool): Trueなら"YYYY-MM-DD"形式、Falseならdate型

    Returns:
        Union[List[str], List[date]]: 祝日のリスト（文字列またはdate型）
    """
    # --- 日付型でなければ変換 ---
    if isinstance(start, str):
        start = datetime.strptime(start, "%Y-%m-%d").date()
    if isinstance(end, str):
        end = datetime.strptime(end, "%Y-%m-%d").date()

    # --- 日付範囲の祝日抽出 ---
    holidays = [
        d
        for d in (start + timedelta(days=i) for i in range((end - start).days + 1))
        if jpholiday.is_holiday(d)
    ]

    return [d.strftime("%Y-%m-%d") for d in holidays] if as_str else holidays

# 日本語フォント設定（存在する最初の候補を適用）
def set_jp_font():
    try:
        import matplotlib.pyplot as plt
        from matplotlib import font_manager
        candidates = ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "TakaoGothic"]
        system_fonts = font_manager.findSystemFonts()
        for cand in candidates:
            for f in system_fonts:
                if cand in f:
                    plt.rcParams["font.family"] = cand
                    return
    except Exception:
        pass  # フォント設定失敗は無視

# SQLite から重量データを読む（テーブル名は推測。存在するテーブルに合わせて変更可）
def load_data_from_sqlite(db_path="/work/app/data/factory_manage/weight_data.db", table_name="weight_data"):
    with sqlite3.connect(db_path) as conn:
        # テーブル存在チェック & 自動選択
        try:
            df_tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
            if table_name not in df_tables['name'].tolist() and len(df_tables):
                # 最初のテーブルを利用
                table_name_local = df_tables['name'].iloc[0]
            else:
                table_name_local = table_name
        except Exception:
            table_name_local = table_name
        df_local = pd.read_sql(f"SELECT * FROM {table_name_local}", conn)
    # 日付カラム推測
    for col in ["伝票日付", "date", "dt"]:
        if col in df_local.columns:
            try:
                df_local[col] = pd.to_datetime(df_local[col])
            except Exception:
                pass
    return df_local

set_jp_font()
print("[INFO] ローカルユーティリティを読み込みました。")

# モジュールキャッシュのクリア（new_model1/new_model2 が部分的に読み込まれている場合に対応）
import sys, importlib
for name in list(sys.modules.keys()):
    if name.startswith('new_model1') or name.startswith('new_model2'):
        del sys.modules[name]
# 足りなければ /works/scripts を先頭に追加してからロード
sys.path.insert(0, '/works/scripts')
try:
    import new_model1
    importlib.reload(new_model1)
    print('reloaded new_model1 from', getattr(new_model1, '__file__', None))
except Exception as e:
    print('reload new_model1 failed:', e)
try:
    import new_model2
    importlib.reload(new_model2)
    print('reloaded new_model2 from', getattr(new_model2, '__file__', None))
except Exception as e:
    print('reload new_model2 failed:', e)

[INFO] ローカルユーティリティを読み込みました。
reloaded new_model1 from /works/scripts/new_model1/__init__.py
reloaded new_model2 from /works/scripts/new_model2/__init__.py
reloaded new_model1 from /works/scripts/new_model1/__init__.py
reloaded new_model2 from /works/scripts/new_model2/__init__.py


In [2]:
import pandas as pd
# from logic.factory_manage.utils.sql import load_data_from_sqlite  # 外部依存 -> ローカル版へ
# from utils.get_holydays import get_japanese_holidays             # 外部依存 -> ローカル版へ

# from utils.font import set_jp_font  # 外部フォント設定 -> ローカル版
# set_jp_font()  # 既に前セルで実行済み

# CSV / DB 読み込み（パスはPRE_HANNNYU配下に限定）
# path = "/works/data/factory_manage/weight_data.db"  # そのまま利用

# df = load_data_from_sqlite(path)
# df["伝票日付"].max()
# df.head()

### 2021年～

In [3]:
# pdは既にCELL INDEX:1でimportされているので、そのまま使えます
df_2021 = pd.read_csv("/works/data/input/2020顧客.csv", encoding="utf-8")
df_2022 = pd.read_csv("/works/data/input/2022顧客.csv", encoding="utf-8")
df_2023 = pd.read_csv("/works/data/input/2023_all.csv", encoding="utf-8")
df_2024 = pd.read_csv("/works/data/input/20240501-20250422.csv", encoding="utf-8")

df_2021 = df_2021[['伝票日付', '商品', '正味重量']]
df_2022 = df_2022[['伝票日付', '商品', '正味重量']]
df_2023 = df_2023[['伝票日付', '商品', '正味重量']]
df_2021.rename(columns={'商品': '品名'}, inplace=True)
df_2022.rename(columns={'商品': '品名'}, inplace=True)
df_2023.rename(columns={'商品': '品名'}, inplace=True)
df_2024 = df_2024[['伝票日付', '品名', '正味重量']]

df_all = pd.concat([df_2021, df_2022, df_2023, df_2024], ignore_index=True)
# 曜日など () を削除
df_all["伝票日付"] = df_all["伝票日付"].str.replace(r"\(.*\)", "", regex=True)
df_all["伝票日付"] = pd.to_datetime(df_all["伝票日付"], format="%Y/%m/%d")
df_all


/tmp/ipykernel_69458/1970061802.py:4: DtypeWarning: Columns (68) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2023 = pd.read_csv("/works/data/input/2023_all.csv", encoding="utf-8")


,伝票日付,品名,正味重量
0,2020-01-04,混合廃棄物A,870.0
1,2020-01-04,混合廃棄物A,450.0
2,2020-01-04,混合廃棄物（焼却物）,2400.0
3,2020-01-04,混合廃棄物A,250.0
4,2020-01-04,混合廃棄物A,800.0
...,...,...,...
184245,2025-05-26,軽量物系A(ｽﾀｲﾛﾌｫｰﾑ),10.0
184246,2025-05-26,廃ﾌﾟﾗｽﾁｯｸ類,30.0
184247,2025-05-26,廃ﾌﾟﾗｽﾁｯｸ類,160.0
184248,2025-05-26,混合廃棄物A,2530.0


### 予約情報

In [4]:
df_reserve = pd.read_csv(
    "/works/data/input/yoyaku_data.csv")
df_reserve["予約日"] = pd.to_datetime(df_reserve["予約日"])
df_reserve.rename(columns={"台数": "予約台数"}, inplace=True)
print(df_reserve["予約日"].min(), df_reserve["予約日"].max())
print(df_reserve.columns)
df_reserve

2023-01-04 00:00:00 2025-05-31 00:00:00
Index(['予約日', '予約得意先名', '固定客', '予約台数'], dtype='object')


,予約日,予約得意先名,固定客,予約台数
0,2023-01-04,アンデス,False,1.0
1,2023-01-04,リサイクルレスキュー,False,1.0
2,2023-01-04,山口興業,False,2.0
3,2023-01-04,明和建装,False,1.0
4,2023-01-04,まごころ清掃社,False,1.0
...,...,...,...,...
45726,2025-05-31,首都高メンテナンス,False,1.0
45727,2025-05-31,シミズオクト,False,1.0
45728,2025-05-31,鈴亀,False,1.0
45729,2025-05-31,鈴木運輸,True,1.0


### 受入番号用

In [5]:
import os
import glob
import pandas as pd

# ディレクトリ内の全CSVファイルパスを取得
csv_dir = "/works/data/input/受入_時刻"
csv_files = glob.glob(os.path.join(csv_dir, "*.csv"))

# 全CSVを読み込んで結合
dfs = []
for f in csv_files:
    df_tmp = pd.read_csv(f)

    # 伝票日付を整形 → 日付型へ変換
    df_tmp["伝票日付"] = df_tmp["伝票日付"].str.replace(r"\(.*?\)", "", regex=True).str.strip()
    df_tmp["伝票日付"] = pd.to_datetime(df_tmp["伝票日付"], format="%Y/%m/%d")

    # 正味重量をカンマ除去して数値化
    df_tmp["正味重量"] = df_tmp["正味重量"].replace({',': ''}, regex=True).astype(float)

    # 受入番号の欠損を埋めて型変換（念のため）
    df_tmp["受入番号"] = df_tmp["受入番号"].fillna(-1).astype(int)

    dfs.append(df_tmp)

df_Ukeire = pd.concat(dfs, ignore_index=True)

# 必要カラムだけ抽出（品名・重量・受入番号）
df_Ukeire = df_Ukeire[["伝票日付", "品名", "正味重量", "受入番号"]].copy()

# 台数カウント準備（日付・品名ごとの受入番号ユニーク数）
df_count = (
    df_Ukeire.groupby(["伝票日付", "品名"])["受入番号"]
    .nunique()
    .reset_index()
    .rename(columns={"受入番号": "台数"})
)

# （参考）結合しておく場合
df_merged = pd.merge(df_Ukeire, df_count, on=["伝票日付", "品名"], how="left")

# リネーム
df_merged.rename(columns={"台数": "搬入済台数"}, inplace=True)

# 結果確認
df_merged


,伝票日付,品名,正味重量,受入番号,搬入済台数
0,2024-12-01,混合廃棄物A,1100.0,31839,25
1,2024-12-01,運搬費,NaN,31839,1
2,2024-12-01,混合廃棄物A,720.0,31851,25
3,2024-12-01,混合廃棄物A,1860.0,31834,25
4,2024-12-01,混合廃棄物A,1060.0,31874,25
...,...,...,...,...,...
64680,2025-03-31,混合廃棄物A,170.0,46735,68
64681,2025-03-31,選別,80.0,46735,13
64682,2025-03-31,GC 軽鉄･ｽﾁｰﾙ類,160.0,46735,9
64683,2025-03-31,金属くず,400.0,46656,1


In [6]:
target_items = ["混合廃棄物A", "混合廃棄物B", "GC 軽鉄･ｽﾁｰﾙ類", "選別", "木くず"]

# 成功・モデル

## 予約数の追加モデル

In [7]:
# new_model1: PRE_HANNNYU/scripts/new_model1 内のローカルモジュールを動的パス追加で利用

import sys, os
SCRIPT_ROOT = "/works/scripts"
if SCRIPT_ROOT not in sys.path:
    sys.path.append(SCRIPT_ROOT)

# robust import

from new_model1 import full_walkforward, ReserveFeatureBuilder


import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt


In [8]:

# default: do not run full heavy pipeline unless explicitly enabled
RUN_PIPELINE = True

# 閾値（暫定的に緩和して予測生成が行われるか確認）
MIN_STAGE1_DAYS = 20  # 元:30
MIN_STAGE2_DAYS = 10  # 元:15

if RUN_PIPELINE:
    # 予約特徴量生成
    df_reserve_feat = ReserveFeatureBuilder(df_reserve).build()

    # 対象日を予約日に限定 (学習データが足りない場合はこのフィルタを緩めることを検討)
    reserve_dates = df_reserve_feat.index
    df_all["伝票日付"] = pd.to_datetime(df_all["伝票日付"])
    df_all = df_all[df_all["伝票日付"].isin(reserve_dates)].copy()

    print("[DEBUG] reservation_filtered_days=", df_all["伝票日付"].nunique())
    print("[DEBUG] reservation_span=", df_all["伝票日付"].min(), "->", df_all["伝票日付"].max())

    # 評価日数
    days_list = [300]
    results = []
    for days in days_list:
        print(f"\n=== {days}日分のデータで評価中 ===")
        latest_date = df_all["伝票日付"].max()
        cutoff_date = latest_date - pd.Timedelta(days=days)
        df_subset = df_all[df_all["伝票日付"] >= cutoff_date].copy()
        hol_max = df_subset["伝票日付"].max()
        hol_min = df_subset["伝票日付"].min()
        holidays = get_japanese_holidays(hol_min, hol_max)
        print("[DEBUG] df_subset_days=", df_subset["伝票日付"].nunique(), "range=", hol_min, "->", hol_max)
        try:
            actual, pred = full_walkforward(
                df_subset,
                holidays=holidays,
                df_reserve=df_reserve,
                min_stage1_days=MIN_STAGE1_DAYS,
                min_stage2_days=MIN_STAGE2_DAYS,
                top_n=2,
            )
            print(f"[DEBUG] returned_lengths actual={len(actual) if isinstance(actual, list) else 'NA'} pred={len(pred) if isinstance(pred, list) else 'NA'}")
            if isinstance(actual, list) and isinstance(pred, list) and len(actual) > 0 and len(pred) > 0:
                r2 = r2_score(actual, pred)
                mae = mean_absolute_error(actual, pred)
                results.append((days, r2, mae))
                print(f"✅ R² = {r2:.3f}, MAE = {mae:,.0f}kg")
            else:
                print("⚠ 評価に十分なデータがありません (予測件数0)")
                results.append((days, None, None))
        except Exception as e:
            print(f"❌ エラー: {e}")
            results.append((days, None, None))

    # 結果表示
    import pandas as pd

    df_result = pd.DataFrame(results, columns=["days", "R2", "MAE"])
    print("\n=== 評価結果 ===")
    print(df_result)
    if df_result["R2"].notna().any():
        plt.plot(df_result["days"], df_result["R2"], marker="o")
        plt.xlabel("Days")
        plt.ylabel("R² Score")
        plt.title("日数別 R² 評価 (new_model1)")
        plt.grid(True)
        plt.show()
else:
    print('Imports successful. To run pipeline, set RUN_PIPELINE = True in this cell.')


[DEBUG] ReserveFeatureBuilder.build: incoming columns=['予約日', '予約得意先名', '固定客', '予約台数']
[DEBUG] ReserveFeatureBuilder.build: sample rows=
{'予約日': [Timestamp('2023-01-04 00:00:00'), Timestamp('2023-01-04 00:00:00'), Timestamp('2023-01-04 00:00:00')], '予約得意先名': ['アンデス', 'リサイクルレスキュー', '山口興業'], '固定客': [False, False, False], '予約台数': [1.0, 1.0, 2.0]}
[DEBUG] reservation_filtered_days= 761
[DEBUG] reservation_span= 2023-01-04 00:00:00 -> 2025-05-26 00:00:00

=== 300日分のデータで評価中 ===
[DEBUG] df_subset_days= 280 range= 2024-07-30 00:00:00 -> 2025-05-26 00:00:00
▶️ full_walkforward 開始
[DEBUG] full_walkforward: df_raw.shape=(47504, 3), holidays_type=<class 'list'> df_reserve.shape=(45731, 4)
[DEBUG] WeightFeatureBuilder.build: past_raw.shape=(47504, 3), target_items=['混合廃棄物A', '混合廃棄物B']
[DEBUG] WeightFeatureBuilder.build: holidays type=<class 'list'>, len_or_none=20
[DEBUG] WeightFeatureBuilder.build: df_pivot.shape=(280, 226), df_feat.shape=(280, 17)
[DEBUG] ReserveFeatureBuilder.build: incoming col

## 天気追加モデル

In [9]:
# new_model2: PRE_HANNNYU/scripts/new_model2 内のローカルモジュール利用 (再読込対応)
import sys, os, importlib
SCRIPT_ROOT = "/works/scripts"
if SCRIPT_ROOT not in sys.path:
    sys.path.append(SCRIPT_ROOT)
# 既存モジュールを再読込して最新パッチ反映
import new_model2.feature_builder as nm2_fb
import new_model2.predict_model_v4_2_4 as nm2_pred
importlib.reload(nm2_fb)
importlib.reload(nm2_pred)
from new_model2.predict_model_v4_2_4 import full_walkforward
from new_model2.feature_builder import WeatherFeatureBuilder, ReserveFeatureBuilder
from sklearn.metrics import r2_score, mean_absolute_error
import pandas as pd
import matplotlib.pyplot as plt

print('[INFO] using WeatherFeatureBuilder from', WeatherFeatureBuilder.__module__)

# 予約 raw データは df_reserve として既に存在 (列: 予約日, 予約得意先名, 固定客, 予約台数)
# 集計済み特徴量はここでは不要なので生成しない（full_walkforward 内で再度 ReserveFeatureBuilder を用いるため）
if not pd.api.types.is_datetime64_any_dtype(df_reserve['予約日']):
    df_reserve['予約日'] = pd.to_datetime(df_reserve['予約日'])

# 日付型保証
if not pd.api.types.is_datetime64_any_dtype(df_all["伝票日付"]):
    df_all["伝票日付"] = pd.to_datetime(df_all["伝票日付"])

# 評価対象の日数リスト
days_list = [90,180,360,720]  # 検証を速くするため一旦 1 パターン
results = []

for days in days_list:
    print(f"\n=== {days}日分のデータで評価中 (weather) ===")
    latest_date = df_all["伝票日付"].max()
    cutoff_date = latest_date - pd.Timedelta(days=days)
    df_subset = df_all[df_all["伝票日付"] >= cutoff_date].copy()
    hol_min = df_subset["伝票日付"].min()
    hol_max = df_subset["伝票日付"].max()
    print('[DEBUG] hol_min, hol_max =', hol_min, hol_max)
    holidays = get_japanese_holidays(hol_min, hol_max)

    # 予約 raw サブセット（列 予約日 でフィルタ）
    mask = (df_reserve['予約日'] >= hol_min) & (df_reserve['予約日'] <= hol_max)
    df_reserve_raw_subset = df_reserve.loc[mask].copy()
    print(f"[DEBUG] reserve_raw_rows={len(df_reserve_raw_subset)} range={df_reserve_raw_subset['予約日'].min()}->{df_reserve_raw_subset['予約日'].max() if len(df_reserve_raw_subset) else 'NA'}")

    # 天気特徴量取得
    weather_builder = WeatherFeatureBuilder(start_date=hol_min, end_date=hol_max, enable_fallback=True)
    df_weather_full = weather_builder.build()
    df_weather = df_weather_full.loc[hol_min:hol_max].copy() if not df_weather_full.empty else df_weather_full
    if len(df_weather) > 0:
        print(f"[DEBUG] weather_rows={len(df_weather)} range={df_weather.index.min()}->{df_weather.index.max()}")
    else:
        print("[DEBUG] weather empty (fallback or no data)")

    try:
        actual, pred = full_walkforward(
            df_raw=df_subset,
            df_reserve=df_reserve_raw_subset,  # 修正: raw を渡す
            holidays=holidays,
            df_weather=df_weather,
            min_stage1_days=30,
            min_stage2_days=15,
            top_n=2,
        )
        print(f"[DEBUG] returned_lengths actual={len(actual) if isinstance(actual,list) else 'NA'} pred={len(pred) if isinstance(pred,list) else 'NA'}")
        if isinstance(actual, list) and isinstance(pred, list) and len(actual) > 0 and len(pred) > 0:
            r2 = r2_score(actual, pred)
            mae = mean_absolute_error(actual, pred)
            results.append((days, r2, mae))
            print(f"✅ R² = {r2:.3f}, MAE = {mae:,.0f}kg")
        else:
            print("⚠ 評価に十分なデータがありません (予測件数0)")
            results.append((days, None, None))
    except Exception as e:
        print(f"❌ エラー: {e}")
        results.append((days, None, None))

# 結果表示
import pandas as pd

df_result = pd.DataFrame(results, columns=["days", "R2", "MAE"])
print("\n=== 評価結果 (new_model2) ===")
print(df_result)
if df_result["R2"].notna().any():
    plt.plot(df_result["days"], df_result["R2"], marker="o")
    plt.xlabel("Days")
    plt.ylabel("R² Score")
    plt.title("日数別 R² 評価 (new_model2)")
    plt.grid(True)
    plt.show()
else:
    print("R² 有効値が無いためプロットをスキップ")

[INFO] using WeatherFeatureBuilder from new_model2.feature_builder

=== 90日分のデータで評価中 (weather) ===
[DEBUG] hol_min, hol_max = 2025-02-25 00:00:00 2025-05-26 00:00:00
[DEBUG] reserve_raw_rows=4773 range=2025-02-25 00:00:00->2025-05-26 00:00:00
[Weather] fetch 2025-02-25 -> 2025-05-26
[Weather] final params start_date=2025-02-25 end_date=2025-05-26
[Weather] status=200 url=https://archive-api.open-meteo.com/v1/archive?latitude=35.6895&longitude=139.6917&start_date=2025-02-25&end_date=2025-05-26&daily=temperature_2m_mean&daily=precipitation_sum&timezone=Asia%2FTokyo
[Weather] rows=91 cols=['平均気温', '降水量', '天気_大雨', '天気_晴れ', '天気_雨', '天気_台風']
[DEBUG] weather_rows=91 range=2025-02-25 00:00:00->2025-05-26 00:00:00
▶️ full_walkforward(new_model2) 開始
[DEBUG] target_items=['混合廃棄物A', '混合廃棄物B']
[DEBUG] feature_list_len=23 (orig=23) df_feat_rows=74
[DEBUG] dates_len=74 min_stage1_days=30 min_stage2_days=15
[SKIP] 2025-03-07 (i=0) < min_stage1_days=30
[SKIP] 2025-03-13 (i=5) < min_stage1_days=30
[SKIP

In [10]:
# デバッグ: hol_min / hol_max の型確認用一時セル
print('sample hol_min/hol_max (from df_all tail)')
print(df_all['伝票日付'].tail(1), type(df_all['伝票日付'].iloc[-1]))

sample hol_min/hol_max (from df_all tail)
184249   2025-05-26
Name: 伝票日付, dtype: datetime64[ns] <class 'pandas._libs.tslibs.timestamps.Timestamp'>


In [11]:
# === 追加セル (このセルを特徴量削減セルより前に新規挿入) ===
# full_walkforward の現在シグネチャに合わせて余分な引数を自動的に除去し
# 戻り値を (actual, pred, model, dates) の4要素に正規化するラッパを直接適用します。
import sys
import inspect
import importlib

# パス設定を確実に行う
SCRIPT_ROOT = "/works/scripts"
if SCRIPT_ROOT not in sys.path:
    sys.path.insert(0, SCRIPT_ROOT)

# モジュールキャッシュをクリア
for name in list(sys.modules.keys()):
    if name.startswith('new_model2'):
        del sys.modules[name]

try:
    import new_model2.predict_model_v4_2_4 as _nm2p
    importlib.reload(_nm2p)
    print("[INFO] new_model2.predict_model_v4_2_4 successfully loaded and reloaded")
except Exception as e:
    print(f"[ERROR] Failed to load new_model2: {e}")
    # フォールバック: 前のセルで既に読み込まれたfull_walkforwardを使用
    from new_model2.predict_model_v4_2_4 import full_walkforward as _original_full_walkforward
    
    def _fw_wrapper(*args, **kwargs):
        # 引数フィルタリングなし（既存のfull_walkforwardをそのまま使用）
        res = _original_full_walkforward(*args, **kwargs)
        # 戻り値正規化
        if not isinstance(res, tuple):
            res = (res,)
        if len(res) == 2:
            actual, pred = res
            model = None
            dates = list(range(len(actual)))
        elif len(res) == 3:
            actual, pred, model = res
            dates = list(range(len(actual)))
        elif len(res) >= 4:
            actual, pred, model, dates = res[:4]
        else:
            actual, pred, model, dates = [], [], None, []
        return actual, pred, model, dates
    
    print("[INFO] Using fallback wrapper")
    # モジュール更新をスキップしてラッパーのみ適用
    import new_model2.predict_model_v4_2_4 as _nm2p
    _nm2p.full_walkforward = _fw_wrapper
    from new_model2.predict_model_v4_2_4 import full_walkforward
    print("[INFO] fallback wrapper installed (always returns 4要素)")
else:
    # 正常ロード時の処理
    _original_full_walkforward = _nm2p.full_walkforward
    _sig = inspect.signature(_original_full_walkforward)
    _supported = set(_sig.parameters.keys())
    print("[INFO] original full_walkforward signature:", _sig)

    def _fw_wrapper(*args, **kwargs):
        # 余分な引数を落とす
        filtered = {k:v for k,v in kwargs.items() if k in _supported}
        dropped = set(kwargs.keys()) - set(filtered.keys())
        if dropped:
            print(f"[WARN] 未サポート引数削除: {dropped}")
        res = _original_full_walkforward(*args, **filtered)
        # 戻り値正規化
        if not isinstance(res, tuple):
            res = (res,)
        if len(res) == 2:
            actual, pred = res
            model = None
            dates = list(range(len(actual)))
        elif len(res) == 3:
            actual, pred, model = res
            dates = list(range(len(actual)))
        elif len(res) >= 4:
            actual, pred, model, dates = res[:4]
        else:
            actual, pred, model, dates = [], [], None, []
        return actual, pred, model, dates

    # モンキーパッチ
    _nm2p.full_walkforward = _fw_wrapper
    from new_model2.predict_model_v4_2_4 import full_walkforward
    print("[INFO] wrapper installed (always returns 4要素)")

[INFO] new_model2.predict_model_v4_2_4 successfully loaded and reloaded
[INFO] original full_walkforward signature: (df_raw, holidays, df_reserve, df_weather, min_stage1_days, min_stage2_days, top_n=5, allowed_features=None)
[INFO] wrapper installed (always returns 4要素)


In [12]:
# === 365日実行結果の詳細分析 ===
print("=== 365日実行結果サマリー ===")
print(f"ベースラインMAE: {base_mae:.4f}")
print(f"予測回数: {len(base_err_df)}回")
print(f"データ期間: {hol_min} → {hol_max}")

if 'imp_all' in locals():
    print(f"\n=== 特徴量重要度トップ10 ===")
    print(imp_all.head(10))
    
    print(f"\n=== 削減候補特徴量（重要度下位5） ===")
    if len(reducible) >= 5:
        print(reducible.tail(5))
    else:
        print("削減候補が5個未満です")

if 'cand_mae' in locals():
    print(f"\n=== 削減テスト結果 ===")
    print(f"削除対象: {to_remove}")
    print(f"性能変化: {base_mae:.4f} → {cand_mae:.4f}")
    print(f"相対増加: {rel_inc*100:.2f}%")
    print(f"統計的有意性: p={p_value:.4f} ({method})")
    print(f"受容判定: {'ACCEPT' if accept else 'REJECT'}")
else:
    print("\n削減テストが未実行または失敗")

print(f"\n[INFO] 365日の長期データによる評価が完了しました")

=== 365日実行結果サマリー ===


NameError: name 'base_mae' is not defined

In [15]:
# --- 改良版 多段階削減ループ (true names) ---
from copy import deepcopy
import random
import math
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

DEF_REL_MAE_TOL = 0.005  # 0.5% 悪化まで許容
DEF_REMOVE_STEP = 1

PROTECT_EXACT = set()
PROTECT_PREFIXES = ("合計",)

random.seed(42)
np.random.seed(42)

# === baseline が未定義ならここで自動生成 ===
_need_baseline = False
for _v in ["base_actual", "base_pred", "base_model", "base_dates"]:
    if _v not in globals():
        _need_baseline = True
        break

if _need_baseline:
    print('[INFO] baseline 未定義 -> 自動計算開始')
    try:
        # 祝日リスト再生成（存在すれば再利用）
        if 'holidays' not in globals():
            hol_min = df_all['伝票日付'].min()
            hol_max = df_all['伝票日付'].max()
            holidays = get_japanese_holidays(hol_min, hol_max)
        # パラメータ既定
        TOP_N = globals().get('TOP_N', 2)
        MIN_STAGE1_DAYS = globals().get('MIN_STAGE1_DAYS', 30)
        MIN_STAGE2_DAYS = globals().get('MIN_STAGE2_DAYS', 15)
        # 天気特徴量存在確認
        if 'df_weather_full' not in globals():
            try:
                from new_model2.feature_builder import WeatherFeatureBuilder
                wb = WeatherFeatureBuilder(start_date=df_all['伝票日付'].min(), end_date=df_all['伝票日付'].max(), enable_fallback=True)
                df_weather_full = wb.build()
                print(f"[INFO] weather built rows={len(df_weather_full)}")
            except Exception as e:
                print('[WARN] 天気特徴量生成失敗 -> なしで継続', e)
                df_weather_full = None
        # full_walkforward import (既にラッパ適用セルがある前提)
        try:
            from scripts.new_model2.predict_model_v4_2_4 import full_walkforward, get_target_items, get_feature_list
        except Exception:
            from new_model2.predict_model_v4_2_4 import full_walkforward, get_target_items, get_feature_list
        base_actual, base_pred, base_model, base_dates = full_walkforward(
            df_all, holidays, df_reserve, df_weather_full, MIN_STAGE1_DAYS, MIN_STAGE2_DAYS,
            top_n=TOP_N, allowed_features=None
        )
        print(f"[INFO] baseline 再計算完了 len={len(base_actual)}")
    except Exception as e:
        print('[ERROR] baseline 自動計算失敗:', e)
        base_actual, base_pred, base_model, base_dates = [], [], None, []

if len(base_actual) == 0:
    print('[ABORT] baseline が空のため特徴量削減をスキップ')
else:
    base_err_df = pd.DataFrame({
        'date': base_dates,
        'actual': base_actual,
        'pred': base_pred
    })
    base_err_df['abs_err'] = (base_err_df['actual'] - base_err_df['pred']).abs()
    base_mae = base_err_df['abs_err'].mean()
    base_r2 = r2_score(base_actual, base_pred) if len(base_actual)>1 else float('nan')
    print(f"[BASE] MAE={base_mae:,.2f} R2={base_r2:.3f} n={len(base_actual)}")

    # 重要度抽出
    imp_true = extract_true_feature_importances(base_model)
    all_features_ordered = imp_true['feature'].tolist()
    print(f"[INFO] feature count={len(all_features_ordered)}")

    # 保護対象除外
    reducible_df = imp_true[~imp_true['feature'].isin(PROTECT_EXACT)]
    for p in PROTECT_PREFIXES:
        reducible_df = reducible_df[~reducible_df['feature'].str.startswith(p)]

    # 重要度小さい順に並べ替え
    reducible_df = reducible_df.sort_values('abs_coef', ascending=True).reset_index(drop=True)

    current_keep = all_features_ordered.copy()
    history = []
    max_steps = 30

    # 安全のため TOP_N など再取得
    TOP_N = globals().get('TOP_N', 2)
    MIN_STAGE1_DAYS = globals().get('MIN_STAGE1_DAYS', 30)
    MIN_STAGE2_DAYS = globals().get('MIN_STAGE2_DAYS', 15)

    for step in range(max_steps):
        cand_remove = []
        for f in reducible_df['feature']:
            if f in current_keep and f not in PROTECT_EXACT and not any(f.startswith(p) for p in PROTECT_PREFIXES):
                cand_remove.append(f)
            if len(cand_remove) >= DEF_REMOVE_STEP:
                break
        if not cand_remove:
            print('[END] 除去候補なし')
            break

        tentative = [f for f in current_keep if f not in cand_remove]

        try:
            original_list = get_feature_list(get_target_items(df_all, TOP_N), extra_features=["天気_晴れ","天気_雨","天気_大雨","天気_台風"])
            tentative = [f for f in tentative if f in original_list]
        except Exception as e:
            print('[WARN] original_list取得失敗 -> スキップ', e)
        if len(tentative) == 0:
            print(f"[SKIP] step={step} 交差後特徴量ゼロ -> 中断")
            break

        print(f"\n[TRY] step={step} remove={cand_remove} -> tentative_len={len(tentative)}")

        cand_actual, cand_pred, cand_model, cand_dates = full_walkforward(
            df_all, holidays, df_reserve, df_weather_full, MIN_STAGE1_DAYS, MIN_STAGE2_DAYS,
            top_n=TOP_N, allowed_features=tentative
        )
        if len(cand_actual) < 3:
            print('[REJECT] データ不足')
            history.append({'step': step, 'removed': cand_remove, 'result': 'REJECT_DATA'})
            break

        cand_err = pd.DataFrame({'actual': cand_actual, 'pred': cand_pred})
        cand_err['abs_err'] = (cand_err['actual'] - cand_err['pred']).abs()
        cand_mae = cand_err['abs_err'].mean()
        cand_r2 = r2_score(cand_actual, cand_pred) if len(cand_actual)>1 else float('nan')

        rel_diff = (cand_mae - base_mae) / base_mae

        base_df_j = base_err_df.copy()
        base_df_j['date'] = base_dates
        cand_df_j = cand_err.copy()
        cand_df_j['date'] = cand_dates
        merged = base_df_j.merge(cand_df_j[['date','abs_err']], on='date', suffixes=('_base','_cand'))
        p_value = np.nan
        if len(merged) >= 5:
            try:
                from scipy.stats import wilcoxon
                stat, p_value = wilcoxon(merged['abs_err_base'], merged['abs_err_cand'])
            except Exception as e:
                print('[WARN] wilcoxon失敗', e)

        accept = (rel_diff <= DEF_REL_MAE_TOL) and (math.isnan(p_value) or p_value > 0.05)

        print(f"[EVAL] cand_mae={cand_mae:,.2f} (diff={rel_diff*100:.2f}%) R2={cand_r2:.3f} p={p_value if not math.isnan(p_value) else 'NA'} -> {'ACCEPT' if accept else 'REJECT'}")

        history.append({
            'step': step,
            'removed': cand_remove,
            'cand_mae': cand_mae,
            'cand_r2': cand_r2,
            'base_mae': base_mae,
            'base_r2': base_r2,
            'rel_diff': rel_diff,
            'p_value': p_value,
            'accept': accept
        })

        if accept:
            current_keep = tentative
            base_actual, base_pred, base_model, base_dates = cand_actual, cand_pred, cand_model, cand_dates
            base_err_df = cand_err.copy()
            base_err_df['date'] = base_dates
            base_mae = cand_mae
            base_r2 = cand_r2
            reducible_df = reducible_df[~reducible_df['feature'].isin(cand_remove)].reset_index(drop=True)
        else:
            PROTECT_EXACT.update(cand_remove)
            reducible_df = reducible_df[~reducible_df['feature'].isin(PROTECT_EXACT)].reset_index(drop=True)

        if len(reducible_df) == 0:
            print('[END] これ以上削減不可')
            break

    print('\n--- 削減履歴 ---')
    try:
        display(pd.DataFrame(history))
    except Exception:
        print(pd.DataFrame(history))

[INFO] baseline 未定義 -> 自動計算開始
▶️ full_walkforward(new_model2) 開始
[DEBUG] target_items=['混合廃棄物A', '混合廃棄物B']
[DEBUG] feature_list_len=23 (orig=23) df_feat_rows=751
[DEBUG] dates_len=751 min_stage1_days=20 min_stage2_days=10
[SKIP] 2023-01-14 (i=0) < min_stage1_days=20
[SKIP] 2023-01-19 (i=5) < min_stage1_days=20
[SKIP] 2023-01-24 (i=10) < min_stage1_days=20
[SKIP] 2023-01-29 (i=15) < min_stage1_days=20

=== 2023-02-03 を予測中 (i=20) ===
[DEBUG] feature_list_len=23 (orig=23) df_feat_rows=751
[DEBUG] dates_len=751 min_stage1_days=20 min_stage2_days=10
[SKIP] 2023-01-14 (i=0) < min_stage1_days=20
[SKIP] 2023-01-19 (i=5) < min_stage1_days=20
[SKIP] 2023-01-24 (i=10) < min_stage1_days=20
[SKIP] 2023-01-29 (i=15) < min_stage1_days=20

=== 2023-02-03 を予測中 (i=20) ===
[DEBUG] ステージ2未実行 rows=1/11

=== 2023-02-04 を予測中 (i=21) ===
[DEBUG] ステージ2未実行 rows=1/11

=== 2023-02-04 を予測中 (i=21) ===
[DEBUG] ステージ2未実行 rows=2/11

=== 2023-02-05 を予測中 (i=22) ===
[DEBUG] ステージ2未実行 rows=2/11

=== 2023-02-05 を予測中 (i=22) ===

NameError: name 'extract_true_feature_importances' is not defined

In [ ]:
# === 最終特徴量サマリー & 再学習・保存ユーティリティ ===
import os, json, pickle, time
from datetime import datetime

# current_keep が存在し、最低限の特徴があるか確認
if 'current_keep' not in globals() or not current_keep:
    print('[FINAL] current_keep が未定義または空のためスキップ')
else:
    print(f'[FINAL] 最終特徴量数: {len(current_keep)}')
    print(current_keep[:30] + (['...'] if len(current_keep) > 30 else []))

    # 祝日と天気を再利用 / 無ければ再生成
    try:
        holidays
    except NameError:
        hol_min = df_all['伝票日付'].min(); hol_max = df_all['伝票日付'].max()
        holidays = get_japanese_holidays(hol_min, hol_max)
    try:
        df_weather_full
    except NameError:
        df_weather_full = None

    # パラメータ取得（存在しなければデフォルト）
    TOP_N = globals().get('TOP_N', 2)
    MIN_STAGE1_DAYS = globals().get('MIN_STAGE1_DAYS', 30)
    MIN_STAGE2_DAYS = globals().get('MIN_STAGE2_DAYS', 15)

    # full_walkforward import（ラッパ適用済み想定）
    try:
        from scripts.new_model2.predict_model_v4_2_4 import full_walkforward
    except Exception:
        from new_model2.predict_model_v4_2_4 import full_walkforward

    print('[FINAL] 最終特徴量で再学習開始')
    t0 = time.time()
    f_actual, f_pred, f_model, f_dates = full_walkforward(
        df_all, holidays, df_reserve, df_weather_full,
        MIN_STAGE1_DAYS, MIN_STAGE2_DAYS,
        top_n=TOP_N, allowed_features=current_keep
    )
    if len(f_actual) == 0:
        print('[FINAL][ERROR] 再学習結果が空')
    else:
        from sklearn.metrics import r2_score, mean_absolute_error
        f_mae = mean_absolute_error(f_actual, f_pred)
        f_r2 = r2_score(f_actual, f_pred) if len(f_actual) > 1 else float('nan')
        print(f'[FINAL] 再学習完了 MAE={f_mae:,.2f} R2={f_r2:.3f} n={len(f_actual)} elapsed={(time.time()-t0):.1f}s')

        # 出力ディレクトリ
        out_dir = '/works/data'
        os.makedirs(out_dir, exist_ok=True)

        # 特徴量リスト保存
        feat_path = os.path.join(out_dir, 'selected_features_final.txt')
        with open(feat_path, 'w', encoding='utf-8') as f:
            for feat in current_keep:
                f.write(feat + '\n')
        print('[FINAL] 特徴量リスト保存:', feat_path)

        # メタ情報 + 成果指標
        meta = {
            'generated_at': datetime.utcnow().isoformat() + 'Z',
            'feature_count': len(current_keep),
            'mae': f_mae,
            'r2': f_r2,
            'n_predictions': len(f_actual),
            'params': {
                'TOP_N': TOP_N,
                'MIN_STAGE1_DAYS': MIN_STAGE1_DAYS,
                'MIN_STAGE2_DAYS': MIN_STAGE2_DAYS,
                'REL_MAE_TOL': DEF_REL_MAE_TOL,
                'REMOVE_STEP': DEF_REMOVE_STEP
            }
        }
        meta_path = os.path.join(out_dir, 'final_model_meta.json')
        with open(meta_path, 'w', encoding='utf-8') as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)
        print('[FINAL] メタ情報保存:', meta_path)

        # モデル保存（stage1_model_dict を pickle）
        try:
            model_path = os.path.join(out_dir, 'final_stage1_model.pkl')
            with open(model_path, 'wb') as f:
                pickle.dump(f_model, f)
            print('[FINAL] モデル保存:', model_path)
        except Exception as e:
            print('[FINAL][WARN] モデル保存失敗:', e)

        # 簡易差分 (baseline との差) があれば表示
        try:
            diff_mae = (f_mae - base_mae) / base_mae * 100
            print(f'[FINAL] baselineとの差: MAE差分={diff_mae:.2f}%')
        except Exception:
            pass
